In [1]:
%load_ext autoreload
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100`% !important; }</style>"))
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=UserWarning)

In [2]:
from apiutils import WebIO
from ioutils import FileIO, HTMLIO
from fileutils import FileInfo, DirInfo
from master import MasterParams, MusicDBPermDir
from pandas import Series, DataFrame, concat
from sys import prefix
from listUtils import getFlatList
from musicdb import PanDBIO
mp    = MasterParams(verbose=True)
io    = FileIO()
wio   = WebIO()
hio   = HTMLIO()
mdbpd = MusicDBPermDir()

MasterBasic()
  ==> ModVals:    100
  ==> Project:    pandb
  ==> MusicDB:    musicdb
MasterPaths()
  ==> Raw:        /Volumes/Piggy/Discog
  ==> Mod:        /Volumes/Seagate/Discog
  ==> Sum:        /Users/tgadfort/Music/Discog
MasterMetas()
  ==> Media:      ['Album', 'SingleEP', 'Appearance', 'Technical', 'Mix', 'Bootleg', 'AltMedia', 'Other']
  ==> Metas:      ['Basic', 'Media', 'Genre', 'Bio', 'Link', 'Metric', 'Counts', 'Dates']
  ==> Searches:   ['Name', 'AlbumMedia', 'SingleEPMedia', 'AppearanceMedia', 'TechnicalMedia', 'MixMedia', 'BootlegMedia', 'AltMediaMedia', 'OtherMedia']
MasterDBs()
  ==> DBs:        ['Discogs', 'Spotify', 'LastFM', 'Genius', 'RateYourMusic', 'MetalArchives', 'Deezer', 'AllMusic', 'MusicBrainz', 'AlbumOfTheYear', 'SetListFM', 'Beatport', 'Traxsource', 'MyMixTapez', 'ClassicalArchives', 'JioSaavn']


In [3]:
from lib import classicalarchives
mio   = classicalarchives.MusicDBIO(verbose=True, mkDirs=False)
webio = classicalarchives.RawWebData()
db    = mio.db
permDBDir = mdbpd.getDBPermPath(db)
print("Saving Perminant {0} DB Data To {1}".format(db, permDBDir.str))

MusicDBBaseDirs(db=ClassicalArchives)
   RawDataDir     = /Volumes/Piggy/Discog/artists-classicalarchives
   ModValDataDir  = /Volumes/Seagate/Discog/artists-classicalarchives-db
   MetaDataDir    = /Volumes/Seagate/Discog/artists-classicalarchives-db/metadata
   SummaryDataDir = /Users/tgadfort/Music/Discog/db-classicalarchives
ClassicalArchives SummaryData
  ==> Basic
  ==> Media
  ==> Genre
  ==> Bio
  ==> Link
  ==> Metric
  ==> Dates
ParseRawDataUtils(mdbdata, mdbdir) [ClassicalArchives]
ClassicalArchives ModValMetaData
  ==> Basic (Universal)
  ==> Media (Media)
  ==> Dates (Universal)
Saving Perminant ClassicalArchives DB Data To /Users/tgadfort/opt/anaconda3/envs/py310/pandb/mdblib/ClassicalArchives


In [4]:
#for modVal in range(78,100):
#    mio.prd.parsePerformerData(modVal, force=True)

In [ ]:
bsdata = hio.get(io.get("/Volumes/Piggy/Discog/artists-classicalarchives/78/performer/1878.p"))
bsdata

In [5]:
from base import MusicDBDir, MusicDBData
permDir = MusicDBDir(permDBDir)
localComposers     = MusicDBData(path=permDir, fname="{0}SearchedForLocalComposers".format(db.lower()))
localPerformers    = MusicDBData(path=permDir, fname="{0}SearchedForLocalPerformers".format(db.lower()))
knownArtists       = {} #mio.data.getSummaryNameData()
searchComposers    = mio.data.getSearchComposersData()
searchPerformers   = mio.data.getSearchPerformersData()
errors             = MusicDBData(path=permDir, fname="{0}SearchedForErrors".format(db.lower()))

In [6]:
##########################################################################################
# Show Summary
##########################################################################################
print("{0} Search Results".format(db))
print("   Local Composers:           {0}".format(len(localComposers.get())))
print("   Local Performers:          {0}".format(len(localPerformers.get())))
print("   Errors:                    {0}".format(len(errors.get())))
print("   Search Composers:          {0}".format(len(searchComposers)))
print("   Search Performers:         {0}".format(len(searchPerformers)))
print("   Known Summary IDs:         {0}".format(len(knownArtists)))

ClassicalArchives Search Results
   Local Composers:           7547
   Local Performers:          18517
   Errors:                    1
   Search Composers:          7547
   Search Performers:         18518
   Known Summary IDs:         0


In [35]:
import os
os.environ["LC_ALL"] = "en_US.UTF-8"
os.environ["LANG"] = "en_US.UTF-8"
#export LC_ALL=en_US.UTF-8; export LANG=en_US.UTF-8

In [39]:
import osascript
def getScript(url, savename, dtime):
    dscript = '''
tell application "Safari"
activate
set URL of document 1 to "{0}"
delay {2}
set myString to source of document 1
end tell
set newFile to POSIX file "{1}"
open for access newFile with write permission
write myString to newFile as «class utf8»
close access newFile
'''.format(url, savename, dtime)
    
    return dscript

# Download ModVal Data

## Download Via OSA

In [ ]:
#url = f"https://www.classicalarchives.com/performers/{ch}.html"
from string import ascii_lowercase
aTypes = ["composers", "performers"]
for aType in aTypes:
    for ch in ascii_lowercase:
        url = f"https://www.classicalarchives.com/{aType}/{ch}.html"
        savename = f"/Users/tgadfort/Documents/code310/pandb/note/classicalarchives/{aType}_{ch}.html"
        if FileInfo(savename).exists():
            continue
        print(f"{url: <60} ==> {savename}")
        dscript  = getScript(url, savename, dtime=15)
        code,out,err = osascript.run(dscript)

## Parse OSA Data

In [ ]:
from lib.classicalarchives import MusicDBID
mdbid = MusicDBID()
aTypes = ["composers", "performers"]
saveData = {}
for aType in aTypes:
    artistData = {}
    files  = DirInfo("/Users/tgadfort/Documents/code310/pandb/note/classicalarchives").glob(f"{aType}_*.html")
    for ifile in files:
        fData = {}
        bsdata = hio.get(open(ifile, encoding="latin-1").read())
        listingDiv = bsdata.find("div", {"class": "listing"})
        if listingDiv:
            refs = listingDiv.findAll("a")
            fData.update({mdbid.get(ref): {"Name": ref.text, "Ref": ref.get('href')} for ref in refs})
        print(len(fData),'\t',ifile)
        artistData.update(fData)
    artistData = DataFrame(artistData).T
    artistData = artistData[artistData["Ref"].apply(lambda ref: isinstance(ref,str))]
    saveData[aType] = artistData

In [ ]:
mio.data.saveSearchComposersData(data=saveData['composers'])
mio.data.saveSearchPerformersData(data=saveData['performers'])

# Download Composer Data

In [8]:
mio   = classicalarchives.MusicDBIO(verbose=False,local=True,mkDirs=False)

In [41]:
localComposers.save(data={})

In [45]:
useSearchData = True
if useSearchData is True:
    composerNames      = searchComposers #.sort_values(by="Num", ascending=False)
    localComposersDict = localComposers.get()
    composerNamesToGet = composerNames[~composerNames.index.isin(localComposersDict.keys())].sample(frac=1)

    print("# {0} Search Results".format(db))
    print("#   Available Names:      {0}".format(len(composerNames)))
    print("#   Known Artist Names:   {0}".format(len(localComposersDict)))
    print("#   Artist Names To Get:  {0}".format(len(composerNamesToGet)))

# ClassicalArchives Search Results
#   Available Names:      7547
#   Known Artist Names:   250
#   Artist Names To Get:  7297


In [43]:
if False:
    localComposersDict  = localComposers.get()
    for i,(composerID,row) in enumerate(composerNamesToGet.iterrows()):
        composerName = row["Name"]
        composerRef  = row["Ref"]
        localComposersDict[composerID] = composerName
        if len(localComposersDict) == 500:
            break

    print("Saving {0} {1} Composers Data".format(len(localComposersDict), db))
    localComposers.save(data=localComposersDict)

In [44]:
from timeutils import Timestat, TermTime
import random

ts = Timestat("Getting {0} composerIDs".format(db))
#tt = TermTime("tomorrow", "10:50am")
tt = TermTime("today", "7:30pm")
maxN = 5000000

n  = 0
localComposersDict  = localComposers.get()
searchedForErrors   = errors.get()
N = composerNamesToGet.shape[0]

for i,(composerID,row) in enumerate(composerNamesToGet.iterrows()):
    composerName = row["Name"]
    composerRef  = row["Ref"]
    if localComposersDict.get(composerID) is not None:
        continue
    if searchedForErrors.get(composerID) is not None:
        continue
        
    url = f"https://www.classicalarchives.com{composerRef}"
    savename = f"/Users/tgadfort/Desktop/ClassicalArchives/Composer/{composerID}.html"
    #if FileInfo(savename).exists():
    #    continue
    print(f"{i+1: >5}/{N: <10}{url: <60} ==> ", end="")
    dscript  = getScript(url, savename, dtime=7+random.randint(0,5))
    code,out,err = osascript.run(dscript)
    if FileInfo(savename).exists():
        print(f"{composerID}")
        localComposersDict[composerID] = composerName
    else:
        searchedForErrors[composerID] = composerName
        print(f"Error in download.")
        
    n += 1
        
    if n % 10 == 0 or n >= maxN:
        print("="*150)
        ts.update(n=n)
        print("Saving {0} {1} Composers Data".format(len(localComposersDict), db))
        localComposers.save(data=localComposersDict)
        if len(searchedForErrors) > 0:
            errors.save(data=searchedForErrors)
        print("="*150)
        webio.wait(5.0)
        if tt.isFinished() or n >= maxN:
            break
    
ts.stop()
print("Saving {0} {1} Composers Data".format(len(localComposersDict), db))
localComposers.save(data=localComposersDict)
if len(searchedForErrors) > 0:
    print("Saving {0} {1} Errors".format(len(searchedForErrors), db))
    errors.save(data=searchedForErrors)

Process [Getting ClassicalArchives composerIDs] Start    ==> Time Is 2022-06-25 18:51:21
========================= termTime(day=today,time=7:30pm) =========================
   ====> Terminate Time Set To 2022-06-25 19:30:00 <====
   ====> Will Terminate Process 38 Minutes and 39 Seconds From Now
    1/7547      https://www.classicalarchives.com/composer/24045.html        ==> 24045
    2/7547      https://www.classicalarchives.com/composer/89916.html        ==> 89916
    3/7547      https://www.classicalarchives.com/composer/55535.html        ==> 55535
    4/7547      https://www.classicalarchives.com/composer/23708.html        ==> 23708
    5/7547      https://www.classicalarchives.com/composer/42261.html        ==> 42261
    6/7547      https://www.classicalarchives.com/composer/10824.html        ==> 10824
    7/7547      https://www.classicalarchives.com/composer/33186.html        ==> 33186
    8/7547      https://www.classicalarchives.com/composer/57897.html        ==> 57897
    9/7

Waiting:   0%|          | 0/100 [00:00<?, ?it/s]

   26/7547      https://www.classicalarchives.com/composer/49493.html        ==> 49493
   27/7547      https://www.classicalarchives.com/composer/70510.html        ==> 70510
   28/7547      https://www.classicalarchives.com/composer/19983.html        ==> 19983
   29/7547      https://www.classicalarchives.com/composer/70332.html        ==> 70332
   30/7547      https://www.classicalarchives.com/composer/3498.html         ==> 3498
   31/7547      https://www.classicalarchives.com/composer/78737.html        ==> 78737
   32/7547      https://www.classicalarchives.com/composer/67849.html        ==> 67849
   33/7547      https://www.classicalarchives.com/composer/15239.html        ==> 15239
   34/7547      https://www.classicalarchives.com/composer/27569.html        ==> 27569
   35/7547      https://www.classicalarchives.com/composer/20427.html        ==> 20427
   36/7547      https://www.classicalarchives.com/composer/45434.html        ==> 45434
   37/7547      https://www.classicalarchive

Waiting:   0%|          | 0/100 [00:00<?, ?it/s]

   51/7547      https://www.classicalarchives.com/composer/33978.html        ==> 33978
   52/7547      https://www.classicalarchives.com/composer/5592.html         ==> 5592
   53/7547      https://www.classicalarchives.com/composer/52864.html        ==> 52864
   54/7547      https://www.classicalarchives.com/composer/48724.html        ==> 48724
   55/7547      https://www.classicalarchives.com/composer/14260.html        ==> 14260
   56/7547      https://www.classicalarchives.com/composer/9787.html         ==> 9787
   57/7547      https://www.classicalarchives.com/composer/34028.html        ==> 34028
   58/7547      https://www.classicalarchives.com/composer/30632.html        ==> 30632
   59/7547      https://www.classicalarchives.com/composer/45303.html        ==> 45303
   60/7547      https://www.classicalarchives.com/composer/51118.html        ==> 51118
   61/7547      https://www.classicalarchives.com/composer/50914.html        ==> 50914
   62/7547      https://www.classicalarchives

...

  201/7547      https://www.classicalarchives.com/composer/46375.html        ==> 46375
  202/7547      https://www.classicalarchives.com/composer/93200.html        ==> 93200
  203/7547      https://www.classicalarchives.com/composer/25586.html        ==> 25586
  204/7547      https://www.classicalarchives.com/composer/17995.html        ==> 17995
  205/7547      https://www.classicalarchives.com/composer/78216.html        ==> 78216
  206/7547      https://www.classicalarchives.com/composer/56596.html        ==> 56596
  207/7547      https://www.classicalarchives.com/composer/73121.html        ==> 73121
  208/7547      https://www.classicalarchives.com/composer/2145.html         ==> 2145
  209/7547      https://www.classicalarchives.com/composer/45984.html        ==> 45984
  210/7547      https://www.classicalarchives.com/composer/25768.html        ==> 25768
  211/7547      https://www.classicalarchives.com/composer/31380.html        ==> 31380
  212/7547      https://www.classicalarchive

Waiting:   0%|          | 0/100 [00:00<?, ?it/s]

  226/7547      https://www.classicalarchives.com/composer/7026.html         ==> 7026
  227/7547      https://www.classicalarchives.com/composer/42236.html        ==> 42236
  228/7547      https://www.classicalarchives.com/composer/10778.html        ==> 10778
  229/7547      https://www.classicalarchives.com/composer/50698.html        ==> 50698
  230/7547      https://www.classicalarchives.com/composer/98588.html        ==> 98588
  231/7547      https://www.classicalarchives.com/composer/2761.html         ==> 2761
  232/7547      https://www.classicalarchives.com/composer/87213.html        ==> 87213
  233/7547      https://www.classicalarchives.com/composer/22935.html        ==> 22935
  234/7547      https://www.classicalarchives.com/composer/61296.html        ==> 61296
  235/7547      https://www.classicalarchives.com/composer/2941.html         ==> 2941
  236/7547      https://www.classicalarchives.com/composer/23718.html        ==> 23718
  237/7547      https://www.classicalarchives.

Waiting:   0%|          | 0/100 [00:00<?, ?it/s]

   ====> Terminate Time [2022-06-25 19:30:00] Is Reached <====
Process [Getting ClassicalArchives composerIDs] Ran For 41 Minutes and 47 Seconds    ==> Time Is 2022-06-25 19:33:08
Saving 250 ClassicalArchives Composers Data
Saving 1 ClassicalArchives Errors


# Download Performer Data

In [ ]:
mio   = classicalarchives.MusicDBIO(verbose=False,local=True,mkDirs=False)

In [ ]:
useSearchData = True

if useSearchData is True:
    performerNames      = searchPerformers #.sort_values(by="Num", ascending=False)
    localPerformersDict = localPerformers.get()
    performerNamesToGet = performerNames[~performerNames.index.isin(localPerformersDict.keys())].sample(frac=1)

    print("# {0} Search Results".format(db))
    print("#   Available Names:      {0}".format(len(performerNames)))
    print("#   Known Artist Names:   {0}".format(len(localPerformersDict)))
    print("#   Artist Names To Get:  {0}".format(len(performerNamesToGet)))
    
#   Artist Names To Get:  18517
#   Artist Names To Get:  14686
#   Artist Names To Get:  9512
#   Artist Names To Get:  5262
#   Artist Names To Get:  4262

In [ ]:
from timeutils import Timestat, TermTime
import random

ts = Timestat("Getting {0} performerIDs".format(db))
tt = TermTime("tomorrow", "6:50am")
#tt = TermTime("today", "10:00pm")
maxN = 5000

n  = 0
localPerformersDict = localPerformers.get()
searchedForErrors   = errors.get()
N = performerNamesToGet.shape[0]

for i,(performerID,row) in enumerate(performerNamesToGet.iterrows()):
    performerName = row["Name"]
    performerRef  = row["Ref"]
    if localPerformersDict.get(performerID) is not None:
        continue
    if searchedForErrors.get(performerID) is not None:
        continue
        
    url = f"https://www.classicalarchives.com{performerRef}"
    savename = f"/Users/tgadfort/Desktop/ClassicalArchives/Performer/{performerID}.html"
    if FileInfo(savename).exists():
        continue
    print(f"{i+1: >5}/{N: <10}{url: <60} ==> ", end="")
    dscript  = getScript(url, savename, dtime=7+random.randint(0,5))
    code,out,err = osascript.run(dscript)
    if FileInfo(savename).exists():
        print(f"{performerID}")
        localPerformersDict[performerID] = performerName
    else:
        searchedForErrors[performerID] = performerName
        print(f"Error in download.")
        
    n += 1
        
    if n % 25 == 0 or n >= maxN:
        print("="*150)
        ts.update(n=n)
        print("Saving {0} {1} performers Data".format(len(localPerformersDict), db))
        localPerformers.save(data=localPerformersDict)
        if len(searchedForErrors) > 0:
            errors.save(data=searchedForErrors)
        print("="*150)
        webio.wait(10.0)
        if tt.isFinished() or n >= maxN:
            break
    
ts.stop()
print("Saving {0} {1} performers Data".format(len(localPerformersDict), db))
localPerformers.save(data=localPerformersDict)
if len(searchedForErrors) > 0:
    print("Saving {0} {1} Errors".format(len(searchedForErrors), db))
    errors.save(data=searchedForErrors)

In [ ]:
localPerformers.save(data=localPerformersDict)

In [ ]:
from lib.classicalarchives import moveLocalFiles, removeLocalFiles
#moveLocalFiles()
removeLocalFiles()
#localPerformers.save(data=localPerformersDict)

In [ ]:
mio.prd.parseComposerData(modVal=1, force=True)
mio.prd.mergeModValData(modVal=1)

# Download & Parse

In [21]:
from fileutils import DirInfo
from ioutils import FileIO
aTypeDir = "Composer"
mioLocal  = DirInfo(f"/Users/tgadfort/Desktop/ClassicalArchives/{aTypeDir}")
io        = FileIO()
print("  ==> Finding Files in {0}: ".format(mioLocal.str), end="")
files = list(mioLocal.glob("*.htm*"))
print("  ==> Found {0} Files".format(len(files)))

  ==> Finding Files in /Users/tgadfort/Desktop/ClassicalArchives/Composer: Running glob(/Users/tgadfort/Desktop/ClassicalArchives/Composer/*.htm*)
  ==> Found 1283 Files


In [40]:
composerID="23701"
composerRef=f"/composer/{composerID}.html"
url = f"https://www.classicalarchives.com{composerRef}"
savename = f"/Users/tgadfort/Desktop/ClassicalArchives/Composer/{composerID}.html"
print(f"{url: <60} ==> ", end="")
dscript  = getScript(url, savename, dtime=3+random.randint(0,5))
code,out,err = osascript.run(dscript)
print(f"{savename}")

https://www.classicalarchives.com/composer/23701.html        ==> /Users/tgadfort/Desktop/ClassicalArchives/Composer/23701.html


In [24]:
names = mio.data.getSummaryNameData()

In [29]:
names[names.index.isin([FileInfo(ifile).basename for ifile in files])].head(30)

36100                              Ron Mazurek
25500                       George David Weiss
61400                           Stefan Safsten
95700                              Simon Wawer
47900                            Edmund Meisel
8700                     Peter Dodds McCormick
87700                             Tom Dempster
3500                          Sergei Vasilenko
63401                           Vladimir Genin
43401                    Raffaele D'Alessandro
56301                            David Behrman
23701                           Carlos Farias
86201                      Wilhelm Baumgartner
42501                    Sigismund von Neukomm
92101                             Charles Lamb
64501                          Alice Hawthorne
43201                              Gerald Near
31001                        Salvatore Messina
34901    Claude - Herbert Schonberg - Kretzmer
33001                      Josef Bardanashvili
36501                      Niccol Castiglioni
96902        

In [33]:
ifile = '/Users/tgadfort/Desktop/ClassicalArchives/Composer/23701.html'
data  = open(ifile, encoding="ascii").read()

UnicodeDecodeError: 'ascii' codec can't decode byte 0x96 in position 286: ordinal not in range(128)

In [32]:
data

'\n\n\n\n\n<!DOCTYPE html PUBLIC "-//W3C//DTD XHTML 1.0 Transitional//EN" "http://www.w3.org/TR/xhtml1/DTD/xhtml1-transitional.dtd">\n<html xmlns="http://www.w3.org/1999/xhtml">\n    <head>\n        <meta http-equiv="Content-Type" content="text/html; charset=utf-8" />\n        <title>Carlos Fari\x96as - Classical Archives</title>\n<meta http-equiv="PICS-Label" content=\'(PICS-1.1 "http://www.classify.org/safesurf/" l gen true for "https://www.classicalarchives.com/" r (SS~~000 1))\'/>\n<meta http-equiv="pics-label" content=\'(pics-1.1 "http://www.icra.org/ratingsv02.html" comment "Single file v2.0" l gen true for "https://www.classicalarchives.com" r (nz 1 vz 1 lz 1 oz 1 cb 1) "http://www.rsac.org/ratingsv01.html" l gen true for "https://www.classicalarchives.com" r (n 0 s 0 v 0 l 0))\'/>\n<meta http-equiv="content-language" content="en"/>\n<meta name="Robots" content="INDEX, FOLLOW"/>\n<meta name="Author" content="Classical Archives LLC"/>\n<meta name="Subject" content="Classical Musi

In [ ]:

        files = list(mioLocal.glob("*.htm*"))
        ts = Timestat("Moving {0} Local Files To Global Directories".format(len(files)))
        for n,ifile in enumerate(files):
            if (n+1) % 25 == 0:
                ts.update(n=n+1,N=len(files))
            dbID    = FileInfo(ifile).basename
            modVal  = mioGlobal.getModVal(dbID)
            dstFile = FileInfo(eval(f"mioGlobal.data.getRaw{aTypeDir}Filename(modVal,dbID)"))

In [ ]:
from utils import PoolIO
pio = PoolIO("ClassicalArchives")
#pio.parse(force=True)
pio.merge()
pio.meta()
pio.sum()
pio.search()

In [ ]:
# Multiple pages
#https://www.classicalarchives.com/artist/6138.html

In [ ]:
from collections import Counter
cntr = Counter()
for modVal in range(100):
    data = mio.data.getModValData(modVal)
    for k,v in data.iteritems():
        for name,vals in v.mediaCounts.counts.items():
            cntr[name] += vals

In [ ]:
cntr.most_common()

In [ ]:
from lib.classicalarchives import RawDBData
rdbData = RawDBData(debug=False)
retval = rdbData.getPerformerData('/Volumes/Piggy/Discog/artists-classicalarchives/4/performer/105604.p')

In [ ]:
retval.show()

In [ ]:
retval.media.media["Performances"][0].get()

In [ ]:
bsdata = hio.get(io.get('/Volumes/Piggy/Discog/artists-classicalarchives/4/performer/105604.p'))

In [ ]:
bsdata

In [ ]:
jsonData['albums']

In [ ]:
jsonLines = [line.strip().split(" = ")[-1] for line in jsData.split("\n")]

In [ ]:
jsonLines

In [ ]:

try:
    jsonData = [json.loads(jsonLine[:-3]) for jsonLine in jsonLines]
except:
    jsonData = []